# Unemployment Rate Profiling
Owner: Anjelika 
Table: `pondolens.dev_bronze.unemployment_rate`  
Baseline: 1,914 rows

**Goal:** identify population records suitable for Q3 and required Silver treatments. All queries are read-only.

**Row meaning:** One row represents one unemployment indicator for a specific geographic area, reference year, and source table version. Each row contains an unemployment estimate, in number of persons or unemployment rate, for a particular location and year.



## 1. Structure and coverage
Inspect columns, sample records, years, and geographic labels. PHILIPPINES is a national total, not a region.


In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.unemployment_rate;

In [0]:
%sql
SELECT *
FROM pondolens.dev_bronze.unemployment_rate
LIMIT 10;

In [0]:
%sql
SELECT COUNT(*)as total_rows
FROM pondolens.dev_bronze.unemployment_rate;

In [0]:
%sql
SELECT 
    year,
    COUNT(*) as row_count,
    COUNT(DISTINCT region) AS geographic_label_count
FROM pondolens.dev_bronze.unemployment_rate
GROUP BY year
ORDER BY year;

In [0]:
%sql
SELECT
    year,
    COUNT(DISTINCT region) AS all_geographic_labels,
    COUNT(DISTINCT CASE
        WHEN UPPER(TRIM(region)) <> 'Philippines'
        THEN region
    END) AS regions_only
FROM pondolens.dev_bronze.unemployment_rate
GROUP BY year
ORDER BY year;

In [0]:
%sql
-- Identify which regions are present in one year but missing in another
WITH all_regions AS (
    SELECT DISTINCT region
    FROM pondolens.dev_bronze.unemployment_rate
    WHERE region IS NOT NULL
),

all_years AS (
    SELECT DISTINCT year
    FROM pondolens.dev_bronze.unemployment_rate
),

region_years AS (
    SELECT DISTINCT region, year
    FROM pondolens.dev_bronze.unemployment_rate
)

SELECT
    r.region,
    y.year,
    CASE
        WHEN ry.region IS NOT NULL THEN 'PRESENT'
        ELSE 'MISSING'
    END AS status
FROM all_regions r
CROSS JOIN all_years y
LEFT JOIN region_years ry
    ON r.region = ry.region
    AND y.year = ry.year
ORDER BY r.region, y.year;

## 2. Data quality
Check required values, numeric/date parsing, and repeated candidate keys. Empty duplicate results mean no repeated keys were found.

In [0]:
%sql
-- checking for nulls
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN region IS NULL
        OR TRIM(region) = '' THEN 1 ELSE 0 END)
        AS missing_region,

    SUM(CASE WHEN year IS NULL
        OR TRIM(CAST(year AS STRING)) = '' THEN 1 ELSE 0 END)
        AS missing_year,

    SUM(CASE WHEN indicator IS NULL
        OR TRIM(indicator) = '' THEN 1 ELSE 0 END)
        AS missing_indicator,

    SUM(CASE WHEN value IS NULL
        OR TRIM(CAST(value AS STRING)) = '' THEN 1 ELSE 0 END)
        AS missing_value,

    SUM(CASE WHEN unit IS NULL
        OR TRIM(unit) = '' THEN 1 ELSE 0 END)
        AS missing_unit,

    SUM(CASE WHEN geographic_level IS NULL
        OR TRIM(geographic_level) = '' THEN 1 ELSE 0 END)
        AS missing_geographic_level,

    SUM(CASE WHEN table_version IS NULL
        OR TRIM(table_version) = '' THEN 1 ELSE 0 END)
        AS missing_table_version

FROM pondolens.dev_bronze.unemployment_rate;

In [0]:
%sql
-- check for numeric and year parsing
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE
        WHEN year IS NOT NULL
        AND TRY_CAST(year AS INT) IS NULL
        THEN 1 ELSE 0 END) AS invalid_year,

    SUM(CASE
        WHEN value IS NOT NULL
        AND TRY_CAST(value AS DOUBLE) IS NULL
        THEN 1 ELSE 0 END) AS invalid_value,

    SUM(CASE
        WHEN standard_error IS NOT NULL
        AND TRIM(CAST(standard_error AS STRING)) <> ''
        AND TRY_CAST(standard_error AS DOUBLE) IS NULL
        THEN 1 ELSE 0 END) AS invalid_standard_error,

    SUM(CASE
        WHEN coefficient_of_variation_percent IS NOT NULL
        AND TRIM(CAST(coefficient_of_variation_percent AS STRING)) <> ''
        AND TRY_CAST(coefficient_of_variation_percent AS DOUBLE) IS NULL
        THEN 1 ELSE 0 END) AS invalid_cv,

    SUM(CASE
        WHEN comparison_year IS NOT NULL
        AND TRIM(CAST(comparison_year AS STRING)) <> ''
        AND TRY_CAST(comparison_year AS INT) IS NULL
        THEN 1 ELSE 0 END) AS invalid_comparison_year,

    SUM(CASE
        WHEN change_from_comparison_year IS NOT NULL
        AND TRIM(CAST(change_from_comparison_year AS STRING)) <> ''
        AND TRY_CAST(change_from_comparison_year AS DOUBLE) IS NULL
        THEN 1 ELSE 0 END) AS invalid_change

FROM pondolens.dev_bronze.unemployment_rate;


In [0]:
%sql
-- check year range
SELECT *
FROM pondolens.dev_bronze.unemployment_rate
WHERE TRY_CAST(year AS INT) IS NULL
   OR TRY_CAST(year AS INT) NOT BETWEEN 2021 AND 2025;

In [0]:
%sql
-- check for repeated candidate business keys
-- region + year + indicator + table_version

SELECT
    geographic_level,
    geographic_name,
    year,
    indicator,
    frequency,
    table_version,
    source_file,
    COUNT(*) AS duplicate_count
FROM pondolens.dev_bronze.unemployment_rate
GROUP BY
    geographic_level,
    geographic_name,
    year,
    indicator,
    frequency,
    table_version,
    source_file
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;

## 3. Reconcile totals
- Reconcile Philippines total against all regions



In [0]:
%sql
WITH unemployment AS (
    SELECT
        TRY_CAST(year AS INT) AS year,
        LOWER(TRIM(geographic_level)) AS geographic_level,
        TRIM(region) AS region,
        TRIM(geographic_name) AS geographic_name,
        table_version,
        TRY_CAST(value AS DOUBLE) AS unemployed
    FROM pondolens.dev_bronze.unemployment_rate
    WHERE indicator = 'Number of unemployed persons'
      AND unit = 'Thousand persons'
),

country_total AS (
    SELECT
        year,
        table_version,
        MAX(unemployed) AS philippines_total
    FROM unemployment
    WHERE geographic_level = 'country'
    GROUP BY year, table_version
),

regional_total AS (
    SELECT
        year,
        table_version,
        COUNT(DISTINCT region) AS region_count,
        SUM(unemployed) AS sum_of_regions
    FROM unemployment
    WHERE geographic_level = 'region'
    GROUP BY year, table_version
)

SELECT
    c.year,
    c.table_version,
    r.region_count,
    c.philippines_total,
    r.sum_of_regions,

    ROUND(
        c.philippines_total - r.sum_of_regions, 3
    ) AS difference,

    CASE
        WHEN r.sum_of_regions IS NULL THEN 'MISSING REGIONS'
        WHEN ABS(c.philippines_total - r.sum_of_regions) <= 0.01
            THEN 'MATCH'
        ELSE 'ISSUE'
    END AS reconciliation_status

FROM country_total c
LEFT JOIN regional_total r
    ON c.year = r.year
    AND c.table_version = r.table_version

ORDER BY c.year, c.table_version;

## 5. Findings and Silver treatments

| Check | Finding | Silver treatment |
|---|---|---|
|Geographic Consistency |Region counts differ across years|Year-specific geographic mapping|
|Historical naming / geographic classification inconsistency|BARMM had ARMM tagging in 2021.|Standardize to BARMM for a harmonized regional label, but retain ARMM as the original source value.|
|Historical administrative boundary change|No Negros Island Region in 2021-2023|Do not create NIR records for 2021–2023 or fill missing values with zero. Preserve the regional boundaries applicable to each year and flag geographic comparability across periods.|
|Inconsistent naming format|Region IV-A from 2024-2025 but RegionIVA with no dash from 2021-2023|Standardize both to Region IV-A across all years. Retain the original source name for traceability.|
|Source version consistency|Four table versions with overlapping years|use data with estimate_status = 'final' for years 2022-2025 and 'not defined' for 2021|


## 6. Assumptions and open checks

| Issue | Assumption |
|---|---|
|ARMM vs. BARMM|ARMM in 2021 is an outdated geographic label referring to BARMM, rather than a separate administrative region.|
|Negros Island Region|The absence of NIR in 2021–2023 reflects the historical administrative structure, rather than missing data.|
|Regional totals|The Philippines total and regional subtotals should reconcile when they use the same geographic boundaries, reporting period, indicator, and source version, subject to rounding and statistical estimation differences.|


## 7. Semantic notes

- Region names reflect the classifications used in their respective source publications and may differ across years.
- ARMM was replaced by BARMM in 2019. The appearance of ARMM in 2021 may reflect outdated source naming and should not automatically be interpreted as a distinct region.
- Changes in administrative boundaries can affect regional unemployment totals and should be considered when interpreting trends.
- Regional unemployment rates cannot be summed to obtain the national unemployment rate. Appropriate labor-force weighting is required.
- Original geographic names and source versions should be preserved to support validation and reproducibility.